# Week 3: SEC 10-K RAG - chunking, embeddings and Pinecone

## Step 0: Install packages

In [81]:
#STEP 0: Install packages and set up Ollama (needed on Colab)
# torch is already installed on Colab. If Colab asks you to restart the session after this cell, do it, then run this cell again.

# 1. Python packages (langchain-ollama added)
%pip install -q langchain-huggingface sentence-transformers langchain-pinecone pinecone langchain-community langchain-text-splitters langchain-experimental langchain-ollama python-dotenv sec-edgar-downloader chardet gradio
# 2. Install the Ollama server
!apt-get install -y -q pciutils zstd
!curl -fsSL https://ollama.com/install.sh | sh

# 3. Start the server in the background and wait until it answers
import subprocess, time, urllib.request

subprocess.Popen(["ollama", "serve"], stdout=open("ollama.log", "w"), stderr=subprocess.STDOUT)
for _ in range(30):
    try:
        urllib.request.urlopen("http://localhost:11434")
        break
    except Exception:
        time.sleep(1)

# 4. Download the model (about 2 GB)
!ollama pull llama3.2


Reading package lists...
Building dependency tree...
Reading state information...
pciutils is already the newest version (1:3.10.0-2build1).
zstd is already the newest version (1.5.5+dfsg2-2build1.1).
0 upgraded, 0 newly installed, 0 to remove and 33 not upgraded.
>>> Cleaning up old version at /usr/local/lib/ollama
>>> Installing ollama to /usr/local
>>> Downloading ollama-linux-amd64.tar.zst
######################################################################## 100.0%
>>> Adding ollama user to video group...
>>> Adding current user to ollama group...
>>> Creating ollama systemd service...
>>> NVIDIA GPU installed.
>>> The Ollama API is now available at 127.0.0.1:11434.
>>> Install complete. Run "ollama" from the command line.



## Step 1: Download the SEC filings

In [82]:
#STEP 1: Downloaded SEC Filings
import os
from sec_edgar_downloader import Downloader

# Colab has no C:\ drive: save to /content/sec there, keep C:\sec when running on Windows
SEC_DIR = "/content/sec" if os.path.exists("/content") else r"C:\sec"

# SEC requires a name and email (any valid email, it's just for their logs)
dl = Downloader("pranusha", "pranuravula0126@example.com", SEC_DIR)

# Download Apple's last 2 annual reports (10-K filings)
# They save to <SEC_DIR>/sec-edgar-filings/AAPL/10-K/
dl.get("10-K", "AAPL", limit=2)

print("✅ Download complete SEC filings")


✅ Download complete SEC filings


## Step 2: Connect to a Llama model (Ollama)

In [83]:
#STEP 2: Connect to llama model since its free
from langchain_ollama import ChatOllama

llm = ChatOllama(model="llama3.2", temperature=0, num_ctx=8192)
response = llm.invoke("Say 'LLM is working' in exactly 3 words.")
print(response.content)

LLM is working.


## Step 3: Fetch the API keys

In [84]:
import os

print("Pinecone key set:", bool(os.environ.get("PINECONE_API_KEY")))
print("HF token set:", bool(os.environ.get("HUGGINGFACEHUB_API_TOKEN")))

Pinecone key set: True
HF token set: True


In [85]:
#STEP 3: Fetch api keys for Pinecone and Huggingface
from getpass import getpass
from dotenv import load_dotenv

load_dotenv()  # reads .env from the current working directory (or a parent folder), if there is one

# On Colab there is no .env file, so ask for the keys (input is hidden and not saved in the notebook)
if not os.environ.get("PINECONE_API_KEY"):
    os.environ["PINECONE_API_KEY"] = getpass("Pinecone API key: ")
if not os.environ.get("HUGGINGFACEHUB_API_TOKEN"):
    os.environ["HUGGINGFACEHUB_API_TOKEN"] = getpass("Hugging Face token: ")

pinecone_api_key = os.environ.get("PINECONE_API_KEY")
huggingface_token = os.environ.get("HUGGINGFACEHUB_API_TOKEN")

# print(pinecone_api_key)
# print(huggingface_token)  # True True, without printing the secrets


## Step 4: Load the embedding model (BGE)

In [86]:
#STEP 4: 
#Create Embeddings to use Embedding Model so that we can store the embeddings in Vector Pinecone DB
import torch
from langchain_huggingface import HuggingFaceEmbeddings

device = "cuda" if torch.cuda.is_available() else "cpu"

embeddings = HuggingFaceEmbeddings(
    model_name="BAAI/bge-base-en-v1.5",
    model_kwargs={"device": device},
    encode_kwargs={"normalize_embeddings": True},
)
print("\nHuggingFaceEmbeddings setup")

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]


HuggingFaceEmbeddings setup


## Step 5: Connect to Pinecone and create the index

In [87]:
#STEP 5
#Connect to Pinecone and create an INDEX
from pinecone import Pinecone, ServerlessSpec

pc = Pinecone(api_key=os.environ["PINECONE_API_KEY"])

INDEX_NAME = "financial-rag"

# Create the index if it doesn't exist yet
if INDEX_NAME not in [idx.name for idx in pc.list_indexes()]:
    pc.create_index(
        name=INDEX_NAME,
        dimension=768,          # matches BAAI/bge-base-en-v1.5
        metric="cosine",
        spec=ServerlessSpec(cloud="aws", region="us-east-1")  # free tier
    )
    print(f"✅ Created index: {INDEX_NAME}")
else:
    print(f"✅ Index already exists: {INDEX_NAME}")

index = pc.Index(INDEX_NAME)
print(index.describe_index_stats())

✅ Index already exists: financial-rag
{'dimension': 768,
 'index_fullness': 0.0,
 'metric': 'cosine',
 'namespaces': {'': {'vector_count': 1142},
                'fixed': {'vector_count': 1198},
                'semantic': {'vector_count': 802}},
 'total_vector_count': 3142,
 'vector_type': 'dense'}


## Step 6: Read the 10-K filings and clean the HTML

In [88]:
#STEP 6
#Read the sek 10 filings and remove the html charecters
import os
import re
import html
from langchain_community.document_loaders import TextLoader

FILINGS_DIR = os.path.join(SEC_DIR, "sec-edgar-filings")   # SEC_DIR is set in Step 1

print("\nRead the sek 10 filings and remove the html charecters")

def clean_edgar_text(text):
    # 0. full-submission.txt bundles many documents: keep only the main 10-K body
    for doc in re.findall(r"<DOCUMENT>(.*?)</DOCUMENT>", text, flags=re.S | re.I):
        if re.search(r"<TYPE>\s*10-K", doc, flags=re.I):
            m = re.search(r"<TEXT>(.*?)</TEXT>", doc, flags=re.S | re.I)
            if m:
                text = m.group(1)
                break

    # 1. drop blocks whose content is not readable text
    text = re.sub(r"<ix:header[^>]*>.*?</ix:header>", " ", text, flags=re.S | re.I)  # hidden XBRL data
    text = re.sub(r"<(script|style)[^>]*>.*?</\1>", " ", text, flags=re.S | re.I)

    # 2. keep paragraph / row breaks before the tags are removed
    text = re.sub(r"</(p|div|tr|li|h[1-6])>|<br\s*/?>", "\n", text, flags=re.I)
    text = re.sub(r"</t[dh]>", " ", text, flags=re.I)

    # 3. remove remaining tags, THEN decode entities (&amp; &#160; &lt; ...)
    text = re.sub(r"<[^>]+>", " ", text)
    text = html.unescape(text)
    text = text.replace("\xa0", " ").replace("\u200b", "")

    # 4. tidy whitespace
    text = re.sub(r"[ \t]+", " ", text)
    text = re.sub(r" ?\n ?", "\n", text)
    text = re.sub(r"(?m)^[ \-_=]+$", "", text)      # lines that are only dashes / underscores
    text = re.sub(r"\n{3,}", "\n\n", text)
    return text.strip()


# ---- 1. find the filing files ----
filing_files = []
for root, dirs, files in os.walk(FILINGS_DIR):
    for file in files:
        if not file.endswith((".htm", ".html", ".txt")):
            continue
        # avoid loading the same filing twice
        if file == "full-submission.txt" and "primary-document.html" in files:
            continue
        path = os.path.join(root, file)
        filing_files.append(path)
        print(path)

print(f"\nTotal files found: {len(filing_files)}")

# ---- 2. load and clean ----
documents = []
for path in filing_files:
    try:
        docs = TextLoader(path, encoding="utf-8", autodetect_encoding=True).load()
        parts = path.split(os.sep)   # ...\AAPL\10-K\0000320193-23-000106\file
        for doc in docs:
            doc.page_content = clean_edgar_text(doc.page_content)
            doc.metadata.update(
                source=path, ticker=parts[-4], form=parts[-3], accession=parts[-2]
            )
        documents.extend(docs)
        print(f"✅ Loaded: {path}  →  {len(docs)} doc(s)")
    except Exception as e:
        print(f"⚠️  Skipped: {path}  →  {e}")

print(f"\nTotal documents loaded: {len(documents)}")
print(documents[0].page_content[:500])



Read the sek 10 filings and remove the html charecters
/content/sec/sec-edgar-filings/AAPL/10-K/0000320193-25-000079/full-submission.txt
/content/sec/sec-edgar-filings/AAPL/10-K/0000320193-26-000026/full-submission.txt

Total files found: 2
✅ Loaded: /content/sec/sec-edgar-filings/AAPL/10-K/0000320193-25-000079/full-submission.txt  →  1 doc(s)
✅ Loaded: /content/sec/sec-edgar-filings/AAPL/10-K/0000320193-26-000026/full-submission.txt  →  1 doc(s)

Total documents loaded: 2
aapl-20250927

UNITED STATES
SECURITIES AND EXCHANGE COMMISSION
Washington, D.C. 20549

FORM 10-K

(Mark One)
☒ ANNUAL REPORT PURSUANT TO SECTION 13 OR 15(d) OF THE SECURITIES EXCHANGE ACT OF 1934
For the fiscal year ended September 27 , 2025
or
☐ TRANSITION REPORT PURSUANT TO SECTION 13 OR 15(d) OF THE SECURITIES EXCHANGE ACT OF 1934
For the transition period from to .
Commission File Number: 001-36743

Apple Inc.
(Exact name of Registrant as specified in its charter)

California 94-2404110
(St


## Step 7: Fixed-size chunks (RecursiveCharacterTextSplitter)

In [89]:
#STEP 7 -- FIXED SIZE CHUNKS
#Split the documents using RecursiveCharacterSplitter but we use chunk size of 512 and overlap of 64
from langchain_text_splitters import RecursiveCharacterTextSplitter

splitter = RecursiveCharacterTextSplitter(chunk_size=512, chunk_overlap=64)
fixed_chunks_raw = splitter.split_documents(documents)

# Filter out noisy chunks
def is_clean_chunk(chunk):
    text = chunk.page_content.strip()
    if len(text) < 100:                          # too short to be useful
        return False
    alpha_ratio = sum(c.isalpha() for c in text) / len(text)
    if alpha_ratio < 0.4:                        # mostly numbers/symbols/whitespace
        return False
    return True

fixed_chunks = [c for c in fixed_chunks_raw if is_clean_chunk(c)]

print(f"Raw chunks:      {len(fixed_chunks_raw)}")
print(f"After filtering: {len(fixed_chunks)}")
print(f"Avg chunk length: {sum(len(c.page_content) for c in fixed_chunks) // len(fixed_chunks)} chars")

print("\n--- Chunk 5 ---")
print(fixed_chunks[5].page_content)
print("\n--- Chunk 6 ---")
print(fixed_chunks[6].page_content)
print("Completed Split the documents using RecursiveCharacterSplitter but we use chunk size of 512 and overlap of 64")


Raw chunks:      1338
After filtering: 1142
Avg chunk length: 393 chars

--- Chunk 5 ---
Indicate by check mark whether the Registrant (1) has filed all reports required to be filed by Section 13 or 15(d) of the Securities Exchange Act of 1934 during the preceding 12 months (or for such shorter period that the Registrant was required to file such reports), and (2) has been subject to such filing requirements for the past 90 days.
Yes ☒ No ☐

--- Chunk 6 ---
Yes ☒ No ☐
Indicate by check mark whether the Registrant has submitted electronically every Interactive Data File required to be submitted pursuant to Rule 405 of Regulation S-T (§232.405 of this chapter) during the preceding 12 months (or for such shorter period that the Registrant was required to submit such files).
Yes ☒ No ☐
Completed Split the documents using RecursiveCharacterSplitter but we use chunk size of 512 and overlap of 64


## Step 8: Embed the fixed chunks and upload to Pinecone

In [90]:
#STEP 8 -- Embed the Fixed Size Chunks and Upload to Pinecone
from langchain_pinecone import PineconeVectorStore

# fixed IDs, so re-running overwrites the same vectors instead of adding duplicates
#ids = [f"{c.metadata['accession']}-{i}" for i, c in enumerate(fixed_chunks)]
fixed_chunk_ids = [f"{c.metadata['accession']}-{i}" for i, c in enumerate(fixed_chunks)]

print(fixed_chunk_ids[:5])   # first 5 IDs

# empty the 'fixed' namespace first so no old vectors are left behind
index.delete(delete_all=True, namespace="fixed")

vectorstore = PineconeVectorStore.from_documents(
    documents=fixed_chunks,
    embedding=embeddings,
    index_name=INDEX_NAME,
    ids=fixed_chunk_ids,
    namespace="fixed",
    batch_size=64,
)

print("Uploaded", len(fixed_chunks), "fixed chunks")

#check that it worked 
import time
time.sleep(10)   # Pinecone's counts can lag a few seconds
print(index.describe_index_stats())   # total_vector_count should match len(chunks)

results = vectorstore.similarity_search("What are the main risk factors?", k=3)
for r in results:
    print(r.metadata["ticker"], r.metadata["accession"])
    print(r.page_content[:300], "\n---")
print("After uplaod searching is completed")

['0000320193-25-000079-0', '0000320193-25-000079-1', '0000320193-25-000079-2', '0000320193-25-000079-3', '0000320193-25-000079-4']
Uploaded 1142 fixed chunks
{'dimension': 768,
 'index_fullness': 0.0,
 'metric': 'cosine',
 'namespaces': {'': {'vector_count': 1142},
                'fixed': {'vector_count': 1142},
                'semantic': {'vector_count': 802}},
 'total_vector_count': 3086,
 'vector_type': 'dense'}
AAPL 0000320193-26-000026
For a discussion of the Company’s cybersecurity-related risks, see Item 1A of this Form 10-K under the heading “Risk Factors.” 
---
AAPL 0000320193-25-000079
For a discussion of the Company’s cybersecurity-related risks, see Item 1A of this Form 10-K under the heading “Risk Factors.” 
---
AAPL 0000320193-26-000026
The Company’s future gross margins can be impacted by a variety of factors, as discussed in Part I, Item 1A of this Form 10-K under the heading “Risk Factors.” As a result, the Company believes, in general, gross margins will be subject 

## Step 9: Semantic chunking

In [91]:
#STEP 9 -- Create SEMANTIC CHUNKS
print("Started Semantic Chunking")
from langchain_core.documents import Document
from langchain_experimental.text_splitter import SemanticChunker

semantic_splitter = SemanticChunker(
    embeddings,
    breakpoint_threshold_type="percentile",
    breakpoint_threshold_amount=65,
)

# try one document first: it embeds every sentence, so it's slow
semantic_chunks = semantic_splitter.split_documents(documents)

lengths = [len(c.page_content) for c in semantic_chunks]
print(f"{len(semantic_chunks)} chunks | min {min(lengths)} | avg {sum(lengths)//len(lengths)} | max {max(lengths)} characters\n")

for i, c in enumerate(semantic_chunks[:3]):
    print(f"--- chunk {i} ({len(c.page_content)} chars) ---")
    print(c.page_content[:400], "\n")
print("Completed Semantic Chunking")

Started Semantic Chunking
802 chunks | min 3 | avg 548 | max 5326 characters

--- chunk 0 (418 chars) ---
aapl-20250927

UNITED STATES
SECURITIES AND EXCHANGE COMMISSION
Washington, D.C. 20549

FORM 10-K

(Mark One)
☒ ANNUAL REPORT PURSUANT TO SECTION 13 OR 15(d) OF THE SECURITIES EXCHANGE ACT OF 1934
For the fiscal year ended September 27 , 2025
or
☐ TRANSITION REPORT PURSUANT TO SECTION 13 OR 15(d) OF THE SECURITIES EXCHANGE ACT OF 1934
For the transition period from to . Commission File Number: 001 

--- chunk 1 (1254 chars) ---
(Exact name of Registrant as specified in its charter)

California 94-2404110
(State or other jurisdiction
of incorporation or organization)
(I.R.S. Employer Identification No.)

One Apple Park Way
Cupertino , California
95014
(Address of principal executive offices) (Zip Code)

( 408 ) 996-1010
(Registrant’s telephone number, including area code)

Securities registered pursuant to Section 12(b) o 

--- chunk 2 (1656 chars) ---
Yes ☐ No ☒

Indicate by check 

## Step 10: Embed the semantic chunks and upload to Pinecone

In [92]:
#STEP 10 -- Embed the Semantic Chunks and Upload to Pinecone    
from langchain_text_splitters import RecursiveCharacterTextSplitter
print("Started Embedding and Uploading Semantic Chunks to Pinecone vector store")
MAX_CHARS = 8000
cap_splitter = RecursiveCharacterTextSplitter(chunk_size=MAX_CHARS, chunk_overlap=100)

final_chunks = []
for chunk in semantic_chunks:
    if len(chunk.page_content) > MAX_CHARS:
        final_chunks.extend(cap_splitter.split_documents([chunk]))   # metadata is copied to each piece
    else:
        final_chunks.append(chunk)

lengths = [len(c.page_content) for c in final_chunks]
print(f"{len(semantic_chunks)} semantic chunks → {len(final_chunks)} final chunks | max {max(lengths)} characters")

sem_ids = [f"{c.metadata['accession']}-sem-{i}" for i, c in enumerate(final_chunks)]

# empty the 'fixed' namespace first so no old vectors are left behind
index.delete(delete_all=True, namespace="semantic")

semantic_store = PineconeVectorStore.from_documents(
    documents=final_chunks,
    embedding=embeddings,
    index_name=INDEX_NAME,
    ids=sem_ids,
    namespace="semantic",
    batch_size=64,
)

print("Uploaded Semantic", len(final_chunks), "chunks")

Started Embedding and Uploading Semantic Chunks to Pinecone vector store
802 semantic chunks → 802 final chunks | max 5326 characters
Uploaded Semantic 802 chunks


## Step 11: Connect to the fixed and semantic vector stores

In [93]:
#STEP 11: get the semantic and fixed vector store 
import os
from pinecone import Pinecone
from langchain_pinecone import PineconeVectorStore
from dotenv import load_dotenv

load_dotenv()
pc = Pinecone(api_key=os.environ.get("PINECONE_API_KEY"))
index = pc.Index(INDEX_NAME)

fixed_vectorstore = PineconeVectorStore(index=index, embedding=embeddings, namespace="fixed")
semantic_vectorstore = PineconeVectorStore(index=index, embedding=embeddings, namespace="semantic")

print(index.describe_index_stats())   # check that both 'fixed' and 'semantic' have vectors

{'dimension': 768,
 'index_fullness': 0.0,
 'metric': 'cosine',
 'namespaces': {'': {'vector_count': 1142},
                'fixed': {'vector_count': 1142},
                'semantic': {'vector_count': 802}},
 'total_vector_count': 3086,
 'vector_type': 'dense'}


In [94]:
#STEP 12: Load the Questions
# These are the kinds of questions a financial analyst would ask
# Mix of specific (numbers) and broad (strategy/risk) questions
test_queries = [
    "What was Apple's total net revenue in fiscal year 2026?",
    "What are the main risk factors related to Apple's supply chain?",
    "How much did Apple spend on research and development?",
    "What is Apple's strategy for services revenue growth?",
    "What were Apple's earnings per share in 2026?",
]

In [95]:
#STEP 13: Define the search without re-ranking
def search_no_rerank(query, store, k=10, accession=None):
    """Plain vector search with scores. Optionally limit it to one filing."""
    filt = {"accession": accession} if accession else None
    return store.similarity_search_with_score(query, k=k, filter=filt)

In [96]:
#STEP 14: Use Re-Ranker to re-rank the output
from sentence_transformers import CrossEncoder

reranker = CrossEncoder("BAAI/bge-reranker-base", device=device, max_length=512)

def search_with_rerank(query, store, fetch_k=20, top_k=3, accession=None):
    """Vector search for fetch_k candidates, then re-rank and keep top_k.
    Optionally limit the search to one filing."""
    filt = {"accession": accession} if accession else None
    docs = store.similarity_search(query, k=fetch_k, filter=filt)          # candidates from Pinecone
    scores = reranker.predict([(query, d.page_content) for d in docs])     # re-score each one
    ranked = sorted(zip(docs, scores), key=lambda x: x[1], reverse=True)
    return ranked[:top_k]

# for query in test_queries:
#     print("=" * 80)
#     print("QUERY:", query)
#     print("=" * 80)

#     for doc, score in search_with_rerank(query, semantic_vectorstore):
#         print(f"\nscore: {score:.3f}")
#         print(doc.page_content[:400])

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

In [97]:
test_queries_apple = [
    # numbers (found in tables)
    "What was Apple's total net sales in fiscal 2026?",
    "How much net sales did iPhone generate in 2026?"
    "What were Services net sales in 2026 compared to 2025?",
    "What was Apple's gross margin percentage in 2026?",
    "What was Apple's net income in 2026?",
    "What were Apple's diluted earnings per share in 2026?",
    "How much did Apple spend on research and development in 2026?",
    "How much cash and marketable securities did Apple hold at the end of 2026?",
    "How much did Apple spend on share repurchases and dividends in 2026?",

    # # explanations (found in paragraphs)
    "Why did Apple's net sales change in 2026 compared to 2025?",
    "What drove the change in Services gross margin in 2026?",
    "What are the main risk factors related to Apple's supply chain?",
    "How do tariffs and trade restrictions affect Apple's business?",
    "What does Apple say about competition in its markets?",
    "How does Apple describe its business strategy and products?",
    "What legal proceedings is Apple involved in?",
    "What does Apple say about its income tax provision and effective tax rate?",
    "Which countries account for the largest share of Apple's net sales?",
]

In [98]:
#STEP 15: Compare Semantic with Vs Without Re-Ranking
ACCESSION = "0000320193-26-000026"      # the 2026 filing

def show(title, results):
    print(f"\n--- {title} ---")
    for rank, (doc, score) in enumerate(results, start=1):
        print(f"\n[{rank}] score: {score:.3f}")
        print(doc.page_content[:400])

for query in test_queries_apple:
    print("=" * 80)
    print("QUERY:", query)
    print("=" * 80)

    no_rerank = search_no_rerank(query, semantic_vectorstore, k=20, accession=ACCESSION)
    reranked = search_with_rerank(query, semantic_vectorstore, fetch_k=20, top_k=3, accession=ACCESSION)

    #In re-ranking or non re-ranking i did not see the first chunk give the correct answer, so generally we give the model top 3 chunks
    show("SEMANTIC - no rerank (top 3 of 10)", no_rerank[:3])
    show("SEMANTIC - reranked (top 3 of 10)", reranked)

QUERY: What was Apple's total net sales in fiscal 2026?

--- SEMANTIC - no rerank (top 3 of 10) ---

[1] score: 0.826
Mac
Mac net sales increased during 2026 compared to 2025 due to higher net sales of laptops. iPad
iPad net sales during 2026 compared to 2025 were relatively flat. Wearables, Home and Accessories
Wearables, Home and Accessories net sales during 2026 compared to 2025 were relatively flat. Services
Services net sales increased during 2026 compared to 2025 primarily due to higher net sales from adver

[2] score: 0.809
Apple Inc. | 2026 Form 10-K | 46

Cash Paid for Income Taxes
Cash paid for income taxes, net in 2026 consisted of the following (in millions):

2026
Federal $ 19,011
State and local 1,669
Foreign:
Ireland 4,956
Singapore 3,253
Other 5,699
Total $ 34,588

Cash paid for income taxes, net was $ 43.4 billion and $ 26.1 billion in 2025 and 2024, respectively. Note 9 – Leases
The Company has lease arran

[3] score: 0.806
Apple Inc. | 2026 Form 10-K | 38

The follow

In [99]:
#STEP 16: Ask the LLM with and without re-ranking
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

prompt_template = ChatPromptTemplate.from_messages([
    ("system", "You are a financial analyst. Answer using ONLY the context below. "
               "Give the exact figure with its unit and year. "
               "If the answer is not in the context, say \"I don't know\"."
               "Always cite specific numbers and figures when available."),
    ("human", "Context:\n{context}\n\nQuestion: {question}"),
])

chain = prompt_template | llm | StrOutputParser()


def ask_llm(query, results):
    """results is a list of (doc, score) pairs."""
    docs = [doc for doc, score in results]
    context = "\n\n---\n\n".join(d.page_content for d in docs)
    answer = chain.invoke({"context": context, "question": query})
    return {"answer": answer, "sources": docs}


for query in test_queries_apple:
    print("=" * 80)
    print("QUERY:", query)
    print("=" * 80)

    no_rerank = search_no_rerank(query, semantic_vectorstore, k=10, accession=ACCESSION)
    reranked  = search_with_rerank(query, semantic_vectorstore, 10, 3, accession=ACCESSION)   # 10 candidates, keep 3

    out_plain    = ask_llm(query, no_rerank[:3])
    out_reranked = ask_llm(query, reranked)

    print("\nNO RERANK :", out_plain["answer"])
    print("\nRERANKED  :", out_reranked["answer"])

QUERY: What was Apple's total net sales in fiscal 2026?

NO RERANK : $478,834 million.

RERANKED  : $478,834 million.
QUERY: How much net sales did iPhone generate in 2026?What were Services net sales in 2026 compared to 2025?

NO RERANK : Here are the answers to your questions:

1. iPhone net sales in 2026 were $254,098 million.
2. Services net sales in 2026 were $123,301 million, compared to $109,158 million in 2025.

RERANKED  : According to the context:

1. iPhone net sales in 2026 were $254,098 million.
2. Services net sales in 2026 were $123,301 million, which is an increase of 13% compared to 2025.
QUERY: What was Apple's gross margin percentage in 2026?

NO RERANK : Total gross margin percentage in 2026 was 48.8%.

RERANKED  : According to the context, Apple's gross margin percentage in 2026 was 48.8%.
QUERY: What was Apple's net income in 2026?

NO RERANK : $132,175 million.

RERANKED  : $132,175 million.
QUERY: What were Apple's diluted earnings per share in 2026?

NO RERANK 

In [100]:
eval_dataset = [
    {
        "question": "What was Apple's total net revenue in fiscal year 2024?",
        "ground_truth": "Apple's total net sales in fiscal year 2024 were $391,035 million."
    },
    {
        "question": "What were Apple's earnings per share in 2024?",
        "ground_truth": "Apple's basic EPS in 2024 was $6.11 and diluted EPS was $6.08."
    },
    {
        "question": "How much did Apple spend on research and development in 2024?",
        "ground_truth": "Apple spent $31,370 million on research and development in fiscal year 2024."
    },
    {
        "question": "What are the main risk factors related to Apple's supply chain?",
        "ground_truth": "Apple's supply chain risks include natural disasters, industrial accidents, public health issues like pandemics, cybersecurity attacks, labor disputes, fire and power shortages, and terrorist attacks that can interrupt manufacturing and delivery of products."
    },
    {
        "question": "What was Apple's net income in fiscal year 2024?",
        "ground_truth": "Apple's net income in fiscal year 2024 was $93,736 million."
    },
    {
        "question": "What was Apple's iPhone revenue in 2024?",
        "ground_truth": "Apple's iPhone revenue in fiscal year 2024 was $201,183 million."
    },
    {
        "question": "What was Apple's services revenue in 2024?",
        "ground_truth": "Apple's services revenue in fiscal year 2024 was $96,169 million, representing approximately 24.6% of total net sales."
    },
    {
        "question": "What was Apple's Mac revenue in fiscal year 2024?",
        "ground_truth": "Apple's Mac revenue in fiscal year 2024 was $29,984 million, a 2% increase from $29,357 million in 2023."
    },
    {
        "question": "What were Apple's Americas net sales in 2024?",
        "ground_truth": "Apple's Americas net sales in fiscal year 2024 were $167,045 million."
    },
    {
        "question": "What were the main factors affecting Apple's gross margin in 2024?",
        "ground_truth": "Apple's gross margin improved in 2024 primarily due to higher services revenue, which carries higher margins than hardware products, along with favorable product mix shifts."
    },
]

print(f"✅ Evaluation dataset ready: {len(eval_dataset)} questions")

✅ Evaluation dataset ready: 10 questions


In [101]:
#STEP 17: Evaluation - compare chunking and reranking
import re, time
import pandas as pd

gold = [
    {"q": "How much net sales did iPhone generate in 2026?",   "a": "254,098"},
    {"q": "What were Services net sales in 2026?",              "a": "123,301"},
    {"q": "What was Apple's total net sales in fiscal 2026?",   "a": "478,834"},
    # TODO: add 7 to 17 more. Take each answer from the 10-K yourself.
]

norm = lambda s: s.replace(",", "")          # "254,098" and "254098" both count


def grounded(answer, results):
    """True if every number in the answer also appears in the retrieved chunks."""
    context = " ".join(d.page_content for d, _ in results)
    numbers = re.findall(r"\d[\d,]*\.?\d*", answer.replace("$", ""))
    missing = [n.rstrip(".,") for n in numbers if n.rstrip(".,") not in context]
    return len(missing) == 0


def run_config(name, retrieve):
    rows = []
    for g in gold:
        start = time.time()
        results = retrieve(g["q"])                              # list of (doc, score)
        answer = ask_llm(g["q"], results)["answer"]
        seconds = time.time() - start
        rows.append({
            "config": name,
            "question": g["q"],
            "expected": g["a"],
            "retrieved": any(norm(g["a"]) in norm(d.page_content) for d, _ in results),
            "answer_correct": norm(g["a"]) in norm(answer),
            "grounded": grounded(answer, results),
            "seconds": round(seconds, 1),
            "answer": answer,
        })
    return rows


configs = {
    "semantic, no rerank": lambda q: search_no_rerank(q, semantic_vectorstore, k=10, accession=ACCESSION)[:3],
    "semantic, rerank":    lambda q: search_with_rerank(q, semantic_vectorstore, 10, 3, accession=ACCESSION),
    "fixed, no rerank":    lambda q: search_no_rerank(q, fixed_vectorstore, k=10, accession=ACCESSION)[:3],
    "fixed, rerank":       lambda q: search_with_rerank(q, fixed_vectorstore, 10, 3, accession=ACCESSION),
}

df = pd.DataFrame([row for name, fn in configs.items() for row in run_config(name, fn)])

summary = df.groupby("config").agg(
    retrieved=("retrieved", "mean"),
    answer_correct=("answer_correct", "mean"),
    grounded=("grounded", "mean"),
    avg_seconds=("seconds", "mean"),
).round(2)

print(summary)
print("\nWrong answers:")
print(df[~df["answer_correct"]][["config", "question", "expected", "answer"]].to_string())

summary.to_csv("eval_summary.csv")
df.to_csv("eval_details.csv", index=False)

                     retrieved  answer_correct  grounded  avg_seconds
config                                                               
fixed, no rerank          0.33            0.33       1.0         0.43
fixed, rerank             1.00            1.00       1.0         0.50
semantic, no rerank       0.67            0.67       1.0         0.47
semantic, rerank          1.00            1.00       1.0         0.83

Wrong answers:
                config                                         question expected                                   answer
1  semantic, no rerank            What were Services net sales in 2026?  123,301                            I don't know.
6     fixed, no rerank  How much net sales did iPhone generate in 2026?  254,098  According to the context, I don't know.
7     fixed, no rerank            What were Services net sales in 2026?  123,301                         $93,943 million.


In [102]:
#STEP 19: A simple UI for asking questions
import gradio as gr

def answer_question(question, use_rerank, chunk_type):
    store = semantic_vectorstore if chunk_type == "semantic" else fixed_vectorstore

    if use_rerank:
        results = search_with_rerank(question, store, 10, 3, accession=ACCESSION)
    else:
        results = search_no_rerank(question, store, k=10, accession=ACCESSION)[:3]

    out = ask_llm(question, results)
    sources = "\n\n---\n\n".join(
        f"[{i}] score {score:.3f}\n{doc.page_content[:500]}"
        for i, (doc, score) in enumerate(results, start=1)
    )
    return out["answer"], sources

demo = gr.Interface(
    fn=answer_question,
    inputs=[
        gr.Textbox(label="Question", placeholder="How much net sales did iPhone generate in 2026?"),
        gr.Checkbox(label="Use re-ranker", value=True),
        gr.Radio(["semantic", "fixed"], value="semantic", label="Chunks"),
    ],
    outputs=[
        gr.Textbox(label="Answer"),
        gr.Textbox(label="Sources used", lines=15),
    ],
    title="Apple 10-K Q&A",
)

demo.launch()

It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://1fcdb5d40644981d2f.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
